# Test: Claude Sonnet — Validator V4 (API)

Anthropic API-based. Default validator alongside V1 and V2.

**Prerequisites:** `ANTHROPIC_API_KEY` set in environment or `.env`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / '.env')

from models.utils import ClaudeSonnet

model = ClaudeSonnet()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'claude-sonnet-4-6',
 'provider': 'anthropic',
 'role': 'validator_4',
 'api_based': True,
 'estimated_cost': '$12-20'}

## 1. Health Check

In [2]:
assert model.health_check(), 'API key invalid or API unreachable!'
print('Health check passed — API key valid')

Health check passed — API key valid


## 2. Chat Completions

In [3]:
messages = [
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 2a. Basic chat
resp = model.chat(messages, system='You are a SOC analyst. Be concise.', max_tokens=100)
print('Basic chat:', resp.content[0].text)

Basic chat: A lateral movement attack is when an adversary, after gaining initial access to a network, moves through systems and resources to expand their reach, escalate privileges, and get closer to their target objective.


In [4]:
# 2b. System message extracted from messages list
msgs_with_system = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]
resp = model.chat(msgs_with_system, max_tokens=100)
print('System extracted:', resp.content[0].text)

System extracted: A lateral movement attack is when an adversary, after gaining initial access to a network, moves through connected systems to expand their reach, escalate privileges, and get closer to high-value targets.


In [5]:
# 2c. Deterministic
resp = model.chat_deterministic(messages, system='You are a SOC analyst.', max_tokens=100)
print('Deterministic:', resp.content[0].text)

Deterministic: A lateral movement attack is when an adversary, after gaining initial access to a network, moves deeper through systems and accounts to expand their foothold, escalate privileges, and reach high-value targets or data.


In [6]:
# 2d. Creative
resp = model.chat_creative(messages, system='You are a SOC analyst.', max_tokens=100)
print('Creative:', resp.content[0].text)

Creative: A lateral movement attack is when an attacker, after gaining initial access to a network, moves through the environment to access additional systems, resources, or data in pursuit of their ultimate objective.


In [7]:
# 2e. Streaming
print('Streaming: ', end='')
with model.chat(messages, system='Be concise.', max_tokens=100, stream=True) as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)
print()

Streaming: A lateral movement attack is when an attacker, after gaining initial access to a network, moves through connected systems to expand their reach, escalate privileges, and access more valuable targets or data.


## 3. Validate (Primary Use Case)

In [8]:
# 3a. Safe proposal — should approve
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement from WS-042 to DC-01 via PsExec. Source 10.0.5.42.'
)
print('Safe proposal:')
print(resp.content[0].text)

Safe proposal:
```json
{
  "approved": true,
  "confidence": 0.87,
  "reasoning": "Isolating WS-042 (10.0.5.42) is an appropriate and proportionate response to confirmed lateral movement activity via PsExec targeting DC-01, a domain controller. PsExec-based lateral movement to a DC is a high-severity indicator of active compromise or adversary progression toward privilege escalation and domain dominance. Network isolation of the source host is a standard, well-established containment action that limits further spread while preserving the host for forensic investigation. The action targets only the confirmed source host and does not affect DC-01 or other systems directly.",
  "risks": [
    "WS-042 may be a shared or critical workstation whose isolation could disrupt legitimate business operations — verify asset criticality before proceeding.",
    "If WS-042 was itself compromised via a prior hop, the true patient-zero host remains uncontained — lateral movement chain should be traced 

In [9]:
# 3b. Dangerous proposal — should reject
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.content[0].text)

Dangerous proposal:
```json
{
  "approved": false,
  "confidence": 0.98,
  "reasoning": "The proposed action is grossly disproportionate to the incident severity. A single phishing email with no evidence of credential compromise does not justify revoking all 500 domain admin credentials. This response would cause massive, unnecessary operational disruption, potentially taking down critical systems and services across the entire organization. The blast radius of the proposed action far exceeds any plausible threat from the described incident. Appropriate response should be scoped to the specific affected user(s) and follow a graduated incident response process.",
  "risks": [
    "Mass service outage: Revoking 500 domain admin accounts simultaneously would likely bring down critical infrastructure, authentication systems, and business operations",
    "Recovery complexity: Re-provisioning 500 accounts creates enormous administrative burden and extended downtime",
    "No threat justific

In [10]:
# 3c. Batch validate
proposals = [
    {'proposal': 'Block IP 10.0.5.12 at firewall.', 'context': 'Confirmed C2 from 10.0.5.12.'},
    {'proposal': 'Delete all firewall rules.', 'context': 'Minor config drift detected.'},
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.content[0].text[:120]}...')


Proposal 0: ```json
{
  "approved": true,
  "confidence": 0.92,
  "reasoning": "Blocking a confirmed C2 (Command and Control) IP add...

Proposal 1: ```json
{
  "approved": false,
  "confidence": 0.99,
  "reasoning": "The proposed action is grossly disproportionate to ...


## 4. Tool Calling (Anthropic Format)

In [11]:
tools = [
    {
        'name': 'query_siem',
        'description': 'Search SIEM logs for security events',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Search query'},
                'time_range': {'type': 'string', 'description': 'Time range'},
            },
            'required': ['query']
        }
    },
    {
        'name': 'isolate_host',
        'description': 'Isolate a host from the network',
        'input_schema': {
            'type': 'object',
            'properties': {
                'hostname': {'type': 'string'},
                'reason': {'type': 'string'},
            },
            'required': ['hostname', 'reason']
        }
    }
]

In [12]:
# 4a. Auto tool choice
tc_messages = [{'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12 in the last hour.'}]
resp = model.tool_call(tc_messages, tools, system='You are a SOC analyst.')
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Tool call: {block.name}({block.input})')
    elif block.type == 'text':
        print(f'Text: {block.text[:80]}')

Tool call: query_siem({'query': 'failed logins src_ip:10.0.5.12', 'time_range': 'last 1 hour'})


In [13]:
# 4b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Required: {block.name}({block.input})')

Required: query_siem({'query': 'failed logins source IP 10.0.5.12', 'time_range': 'last 1 hour'})


In [14]:
# 4c. Specific tool
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Specific: {block.name}({block.input})')

Specific: isolate_host({'hostname': '10.0.5.12', 'reason': 'Suspicious failed login activity detected'})


## 5. Structured Output (JSON)

In [15]:
json_messages = [
    {'role': 'user', 'content': 'Classify: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.content[0].text)

JSON mode: ```json
{
  "severity": "medium",
  "category": "brute_force_attack",
  "confidence": 0.92
}
```


## 6. Batch Chat

In [16]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.content[0].text}')

Batch 0: Phishing is a cyberattack where criminals impersonate legitimate entities (such as banks or companies) through deceptive emails, messages, or websites to trick people into revealing sensitive information like passwords or financial details.
Batch 1: Ransomware is a type of malicious software that encrypts a victim's files or locks their system, demanding a ransom payment in exchange for restoring access.


## 7. Token Usage

In [17]:
resp = model.chat(messages, max_tokens=100)
print(f'Input tokens:  {resp.usage.input_tokens}')
print(f'Output tokens: {resp.usage.output_tokens}')

Input tokens:  17
Output tokens: 43


## 8. Get Config

In [18]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "claude-sonnet-4-6",
  "provider": "anthropic",
  "role": "validator_4",
  "api_based": true,
  "estimated_cost": "$12-20"
}


## Summary

All tests passed if no cells raised exceptions above.